# 01 -- Setup & Data Download

**Environment: Google Colab (T4 GPU)**

| What | Where |
|---|---|
| Source code | `/content/drive/MyDrive/Ai_TRAINING/COMP2050_Project` |
| Dataset (Kvasir-SEG) | `.../data/Kvasir-SEG/` on Drive (persistent) |
| Local data (fast reads) | `/content/data/` (VM SSD, copied from Drive) |

**Prerequisites:** Run this notebook once to set up data. After that, skip to `02_train.ipynb`.

**Steps:**
1. Mount Drive & install dependencies
2. Clone repo to Drive (first time only)
3. Download Kvasir-SEG to Drive
4. Verify dataset & imports

---
## 0 - Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
%pip install -q segmentation-models-pytorch scipy scikit-learn pandas tqdm matplotlib
print("Install done")

In [ ]:
import os, shutil, sys
from pathlib import Path
import torch

# -- Paths --
DRIVE_BASE = Path("/content/drive/MyDrive/Ai_TRAINING")
REPO_ROOT  = DRIVE_BASE / "COMP2050_Project"
DRIVE_DATA = REPO_ROOT / "data"

# Local VM (fast SSD)
LOCAL_DATA = Path("/content/data")

print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

---
## 1 - Clone repo to Drive (first time only)

If the repo already exists on Drive, this cell skips the clone.

In [ ]:
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists():
    print(f"Repo already on Drive: {REPO_ROOT}")
else:
    print("Cloning repo to Drive...")
    !git clone https://github.com/duongphamminhdung/COMP2050_Project.git {REPO_ROOT}
    print(f"Cloned to: {REPO_ROOT}")

# Add to Python path
sys.path.insert(0, str(REPO_ROOT))

# Verify imports
from models import get_model
from losses import get_loss
from data.dataset import KvasirSEGDataset
from utils.metrics import compute_all_metrics
print("All modules imported")

---
## 2 - Download Kvasir-SEG to Drive

Downloads to Drive so it persists across sessions.

In [ ]:
# Setup Kaggle credentials
import json
from google.colab import userdata

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

token = userdata.get('KAGGLE_API_TOKEN')
username = 'dngdngphmminh'

creds = {"username": username, "key": token}
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print("Kaggle credentials configured")

In [ ]:
DRIVE_DATA.mkdir(parents=True, exist_ok=True)

# Check if already downloaded
images_dir = DRIVE_DATA / "Kvasir-SEG" / "images"
if images_dir.exists() and len(list(images_dir.glob("*.*"))) > 0:
    n_images = len(list(images_dir.glob("*.*")))
    n_masks = len(list((DRIVE_DATA / "Kvasir-SEG" / "masks").glob("*.*")))
    print(f"Dataset already on Drive: {n_images} images, {n_masks} masks")
else:
    print("Downloading Kvasir-SEG to Drive...")
    !pip install -q kaggle
    !kaggle datasets download -d debeshjha/kvasirseg -p {DRIVE_DATA} --unzip
    n_images = len(list((DRIVE_DATA / "Kvasir-SEG" / "images").glob("*.*")))
    print(f"Downloaded: {n_images} images")

---
## 3 - Verify dataset & model forward pass

In [ ]:
# Test dataset loading from Drive
from data.dataset import KvasirSEGDataset

train_ds = KvasirSEGDataset(str(DRIVE_DATA), split='train', seed=42)
val_ds = KvasirSEGDataset(str(DRIVE_DATA), split='val', seed=42)
test_ds = KvasirSEGDataset(str(DRIVE_DATA), split='test', seed=42)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# Visualize 3 samples
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    img, mask = train_ds[i]
    img_np = img.numpy().transpose(1, 2, 0)
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    mask_np = mask.squeeze().numpy()
    axes[0, i].imshow(img_np); axes[0, i].set_title(f'Sample {i+1}'); axes[0, i].axis('off')
    axes[1, i].imshow(mask_np, cmap='gray'); axes[1, i].set_title(f'Mask {i+1}'); axes[1, i].axis('off')
plt.suptitle('Kvasir-SEG Dataset Samples'); plt.tight_layout(); plt.show()

In [ ]:
# Verify all 4 models work
from models import get_model

for name in ["unet", "attention_unet", "resunet", "smp_resnet18"]:
    model = get_model(name)
    x = torch.randn(1, 3, 256, 256)
    y = model(x)
    params = sum(p.numel() for p in model.parameters())
    print(f"{name:20s}: {tuple(x.shape)} -> {tuple(y.shape)}, {params:,} params")

In [ ]:
print("Setup complete. Ready to train.")
print(f"\nNext step: open 02_train.ipynb")